# Computation C — composition laws: fusion, anti-speciality, the baryon

**It from Bit via Gödel, Paper 1** · companion to `app:qiskit-composition` · the executable face of the Coecke–Kissinger classification

Three experiments. **(1)** GHZ$_3 \circ$ GHZ$_3$ = GHZ$_4$ exactly (spider fusion). **(2)**
W$_3 \circ$ W$_3$ has *zero* overlap with W$_4$ (anti-speciality). **(3)** A closed **triangle**
of GHZ colour edges carries exactly GHZ$_3$ on its free legs — *the baryon is the wiring* — and
the fully closed loop has singlet weight 1.

On hardware, the Bell cap $\langle\Phi^+|$ becomes a **Bell-basis measurement**
(`CX` + `H`, measure, postselect on `00`). Postselection rates: fusion $1/4$ (~2048 of 8192
shots), triangle $1/32$ (~256 shots). A Pauli-frame correction could recover all outcomes — left
as the natural exercise. Witnesses on the postselected register: the Z-basis distribution
(population) and the $\langle X^{\otimes n}\rangle$ parity (coherence), the standard GHZ pair.

In [1]:
USE_HARDWARE = False   # flip to True to run on IBM hardware
SHOTS = 8192

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

Wvec = np.zeros(8, complex); Wvec[0b001] = Wvec[0b010] = Wvec[0b100] = 1/np.sqrt(3)

def ghz_on(qc, qs):
    qc.h(qs[0]); qc.cx(qs[0], qs[1]); qc.cx(qs[1], qs[2])

def cap(qc, a, b):          # Bell measurement basis change; postselect 00 later
    qc.cx(a, b); qc.h(a)

def fusion_circuit(kind):   # 6 qubits, cap on (q2, q3)
    qc = QuantumCircuit(6)
    if kind == "ghz":
        ghz_on(qc, [0, 1, 2]); ghz_on(qc, [3, 4, 5])
    else:
        qc.prepare_state(Wvec, [0, 1, 2]); qc.prepare_state(Wvec, [3, 4, 5])
    cap(qc, 2, 3)
    return qc

def triangle_circuit():     # 9 qubits, ring caps (2,3), (5,6), (8,0); free legs 1,4,7
    qc = QuantumCircuit(9)
    for qs in ([0,1,2], [3,4,5], [6,7,8]):
        ghz_on(qc, qs)
    cap(qc, 2, 3); cap(qc, 5, 6); cap(qc, 8, 0)
    return qc

## Exact baseline: postselect the statevector, reproduce the appendix numbers

In [2]:
def postselected(qc, zero_qubits, keep_qubits):
    sv = Statevector(qc).data
    n = qc.num_qubits
    out = {}
    for i, a in enumerate(sv):
        if abs(a) < 1e-12: continue
        bits = [(i >> q) & 1 for q in range(n)]
        if any(bits[q] for q in zero_qubits): continue
        key = sum(bits[q] << k for k, q in enumerate(keep_qubits))
        out[key] = out.get(key, 0) + a
    vec = np.zeros(2**len(keep_qubits), complex)
    for k, a in out.items(): vec[k] = a
    w = float(np.vdot(vec, vec).real)
    return vec/np.sqrt(w), w

def fid(v, target):
    return abs(np.vdot(target, v))**2

g4 = np.zeros(16, complex); g4[0] = g4[15] = 1/np.sqrt(2)
w4 = np.zeros(16, complex)
for k in range(4): w4[1 << k] = 0.5
g3 = np.zeros(8, complex); g3[0] = g3[7] = 1/np.sqrt(2)

v, w = postselected(fusion_circuit("ghz"), [2, 3], [0, 1, 4, 5])
print(f"GHZ fusion : fidelity with GHZ4 = {fid(v, g4):.6f}  (weight {w:.4f})")
v, w = postselected(fusion_circuit("w"), [2, 3], [0, 1, 4, 5])
print(f"W fusion   : fidelity with W4   = {fid(v, w4):.6f}  (weight {w:.4f})")
v, w = postselected(triangle_circuit(), [0, 2, 3, 5, 6, 8], [1, 4, 7])
print(f"GHZ triangle: fidelity with GHZ3 = {fid(v, g3):.6f}  (weight {w:.4f})")
print(f"closed-loop singlet weight <GHZ3|triangle> = {abs(np.vdot(g3, v)):.6f}")

GHZ fusion : fidelity with GHZ4 = 1.000000  (weight 0.2500)
W fusion   : fidelity with W4   = 0.000000  (weight 0.2778)
GHZ triangle: fidelity with GHZ3 = 1.000000  (weight 0.0312)
closed-loop singlet weight <GHZ3|triangle> = 1.000000


## Sampled run — five pubs, one job

Per experiment: a Z-basis circuit (population) and, for the GHZ outputs, an X-basis circuit
(parity/coherence). Postselection on the cap bits happens in counts processing.

In [3]:
def with_meas(qc, x_basis_on=()):
    out = qc.copy()
    for q in x_basis_on: out.h(q)
    out.measure_all()
    return out

jobs = {
  "fusion_ghz_Z": (with_meas(fusion_circuit("ghz")), [2,3], [0,1,4,5]),
  "fusion_ghz_X": (with_meas(fusion_circuit("ghz"), (0,1,4,5)), [2,3], [0,1,4,5]),
  "fusion_w_Z"  : (with_meas(fusion_circuit("w")), [2,3], [0,1,4,5]),
  "triangle_Z"  : (with_meas(triangle_circuit()), [0,2,3,5,6,8], [1,4,7]),
  "triangle_X"  : (with_meas(triangle_circuit(), (1,4,7)), [0,2,3,5,6,8], [1,4,7]),
}
circ_list = [v[0] for v in jobs.values()]

if USE_HARDWARE:
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
    service = QiskitRuntimeService()
    backend = service.least_busy(simulator=False, operational=True)
    print("backend:", backend.name)
    pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
    result = Sampler(mode=backend).run([pm.run(c) for c in circ_list], shots=SHOTS).result()
else:
    from qiskit.primitives import StatevectorSampler
    result = StatevectorSampler().run(circ_list, shots=SHOTS).result()

def post_counts(counts, zero_qubits, keep_qubits):
    out, kept = {}, 0
    for b, c in counts.items():
        if any(b[-1-q] == '1' for q in zero_qubits): continue
        kept += c
        key = ''.join(b[-1-q] for q in reversed(keep_qubits))
        out[key] = out.get(key, 0) + c
    return out, kept

processed = {}
for (name, (qc, zq, kq)), res in zip(jobs.items(), result):
    counts = res.data.meas.get_counts()
    pc, kept = post_counts(counts, zq, kq)
    processed[name] = (pc, kept)
    ideal = {"fusion_ghz": 0.25, "fusion_w": 5/18, "triangle": 1/32}[name.rsplit("_", 1)[0]]
    print(f"{name:13s} postselected {kept}/{SHOTS} shots "
          f"({kept/SHOTS:.3f}; ideal {ideal:.3f})")

fusion_ghz_Z  postselected 2043/8192 shots (0.249; ideal 0.250)
fusion_ghz_X  postselected 1951/8192 shots (0.238; ideal 0.250)
fusion_w_Z    postselected 2331/8192 shots (0.285; ideal 0.278)
triangle_Z    postselected 258/8192 shots (0.031; ideal 0.031)
triangle_X    postselected 251/8192 shots (0.031; ideal 0.031)


In [4]:
def show(name, expect):
    pc, kept = processed[name]
    dist = {k: round(v/kept, 3) for k, v in sorted(pc.items())}
    print(f"{name}: {dist}\n   expected: {expect}")

def parity(name):
    pc, kept = processed[name]
    return sum((-1)**k.count('1') * v for k, v in pc.items()) / kept

show("fusion_ghz_Z", "0000 and 1111, each ~0.5  (GHZ4 population)")
print(f"fusion_ghz_X parity <XXXX> = {parity('fusion_ghz_X'):+.4f}   (ideal +1)\n")
show("fusion_w_Z", "0000, 0101, 0110, 1001, 1010 each ~0.2; ZERO on W4's single-excitation strings")
print()
show("triangle_Z", "000 and 111, each ~0.5  (the baryon: GHZ3 on the free legs)")
print(f"triangle_X parity <XXX> = {parity('triangle_X'):+.4f}   (ideal +1)")

fusion_ghz_Z: {'0000': 0.494, '1111': 0.506}
   expected: 0000 and 1111, each ~0.5  (GHZ4 population)
fusion_ghz_X parity <XXXX> = +1.0000   (ideal +1)

fusion_w_Z: {'0000': 0.194, '0101': 0.215, '0110': 0.187, '1001': 0.212, '1010': 0.192}
   expected: 0000, 0101, 0110, 1001, 1010 each ~0.2; ZERO on W4's single-excitation strings

triangle_Z: {'000': 0.527, '111': 0.473}
   expected: 000 and 111, each ~0.5  (the baryon: GHZ3 on the free legs)
triangle_X parity <XXX> = +1.0000   (ideal +1)


## Reading

Fusion holds for GHZ and fails for W with the specific vacuum-plus-double-excitation signature —
anti-speciality as an inspectable distribution. The triangle's free legs carry GHZ$_3$: *the
statement that a quark's C-qubit is GHZ-entangled with its partners in a hadron is an output of
the composition law, not an assumption about baryons.* These are the primitive moves for the
ZX-calculus organisation of the closed-loop (glueball) spectrum, `sec:predictions`.

Hardware notes: the triangle is 9 qubits with three GHZ preps and three Bell measurements —
shallow, but the $1/32$ postselection leaves ~256 events at 8192 shots, so raise `SHOTS` (or
implement the Pauli-frame correction) for tight parities. Expect $\langle X^{\otimes n}\rangle$
to degrade fastest; it is the coherence witness.